# 22 · Réseaux récurrents (RNN, LSTM, GRU) — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch22_rnn/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch22_rnn/03_notebook.ipynb`, créée par `python tools/start_chapter.py 22`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 22.Q1 à 22.Q12, 22.R1 à 22.R3, 22.1 à 22.5, 22.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 22.12 à 22.14 | ta librairie sequences.py : vocabulaire, encodage et fenêtres ; un MLP sur des fenêtres mélangées (*prochaine session de génération*) | | | |
| B | 22.15 à 22.19 | ta librairie rnn.py : les cellules RNN, LSTM et GRU, comparées à PyTorch ; la disparition du gradient mesurée avec autograd (*prochaine session de génération*) | | | |
| C | 22.20 à 22.24 | l'échantillonnage avec température, une série temporelle piégée, les formes des RNN profonds et bidirectionnels, les taches solaires face à la persistance (*prochaine session de génération*) | | | |
| D | 22.25 à 22.29 | le générateur de Holmes, sa loss et sa perplexité, Holmes contre Verne, le CNN-LSTM, le défi des taches solaires (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="22")  # your own library

## Objectifs et rappel express

- Découper une série ou un texte en fenêtres, compter les exemples, et valider une prévision sans fuite de données.
- Programmer en NumPy, dans `mylearn.rnn`, les cellules RNN, LSTM et GRU, et les comparer à `nn.RNN`, `nn.LSTM` et `nn.GRU`.
- Mesurer la disparition du gradient, puis entraîner un LSTM qui prévoit les taches solaires et un autre qui écrit du Holmes caractère par caractère.

**Rappel express.** Cellule RNN : $h_t = \tanh(x_t W_{ih} + b_{ih} + h_{t-1} W_{hh} + b_{hh})$. LSTM (portes $i, f, g, o$) : $c_t = f_t \odot c_{t-1} + i_t \odot g_t$, $h_t = o_t \odot \tanh(c_t)$. GRU (portes $r, z, n$) : $h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$. Paramètres d'une couche PyTorch : $k\,(DH + H^2 + 2H)$, $k = 1, 4, 3$. Fenêtres : $N = \lfloor (T - w - h)/s \rfloor + 1$. Température : $p = \mathrm{softmax}(z/T)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 22.1 à 22.5 et 🧮 22.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme : entre guillemets (`"(3, 5, 1)"`) ; un mot : entre guillemets (`"fit"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 22.6 et 22.7, la réflexion (🗣️ 22.9, ⚖️ 22.10, 📄 22.11) et l'entretien se corrigent avec `05_solutions.md`.

**22.Q1 — Pourquoi un réseau dense ignore l'ordre**

In [ ]:
# 22.Q1: the values you found on paper
answer_22_Q1a = ...  # a) True or False
answer_22_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_22_Q1c = ...  # c) True or False
answer_22_Q1d = ...  # d) the letter of your choice, e.g. "E"
answer_22_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_22_Q1a, answer_22_Q1b, answer_22_Q1c, answer_22_Q1d, answer_22_Q1e]):
    wb.check(f"22.Q1{letter}", answer)

**22.Q2 — État ou poids : qu'est-ce qui change en production ?**

In [ ]:
# 22.Q2: the values you found on paper
answer_22_Q2a = ...  # a) True or False
answer_22_Q2b = ...  # b) True or False
answer_22_Q2c = ...  # c) True or False
answer_22_Q2d = ...  # d) True or False
answer_22_Q2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_22_Q2a, answer_22_Q2b, answer_22_Q2c, answer_22_Q2d, answer_22_Q2e]):
    wb.check(f"22.Q2{letter}", answer)

**22.Q3 — Délai, déroulement et pas de temps**

In [ ]:
# 22.Q3: the values you found on paper
answer_22_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_22_Q3b = ...  # b) a whole number
answer_22_Q3c = ...  # c) a whole number
answer_22_Q3d = ...  # d) True or False
answer_22_Q3e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_22_Q3a, answer_22_Q3b, answer_22_Q3c, answer_22_Q3d, answer_22_Q3e]):
    wb.check(f"22.Q3{letter}", answer)

**22.Q4 — Entrée one-hot, taille de l'état et sens de ses valeurs**

In [ ]:
# 22.Q4: the values you found on paper
answer_22_Q4a = ...  # a) a whole number
answer_22_Q4b = ...  # b) a whole number
answer_22_Q4c = ...  # c) True or False
answer_22_Q4d = ...  # d) a whole number
answer_22_Q4e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_22_Q4a, answer_22_Q4b, answer_22_Q4c, answer_22_Q4d, answer_22_Q4e]):
    wb.check(f"22.Q4{letter}", answer)

**22.Q5 — (échantillons, pas de temps, features) : la bonne organisation**

In [ ]:
# 22.Q5: the values you found on paper
answer_22_Q5a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_22_Q5b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_22_Q5c = ...  # c) True or False
answer_22_Q5d = ...  # d) the letter of your choice, e.g. "E"
answer_22_Q5e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_22_Q5a, answer_22_Q5b, answer_22_Q5c, answer_22_Q5d, answer_22_Q5e]):
    wb.check(f"22.Q5{letter}", answer)

**22.Q6 — BPTT, gradients évanescents et explosifs, dépendances longues**

In [ ]:
# 22.Q6: the values you found on paper
answer_22_Q6a = ...  # a) the letter of your choice, e.g. "E"
answer_22_Q6b = ...  # b) the letter of your choice, e.g. "E"
answer_22_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_22_Q6d = ...  # d) True or False
answer_22_Q6e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_22_Q6a, answer_22_Q6b, answer_22_Q6c, answer_22_Q6d, answer_22_Q6e]):
    wb.check(f"22.Q6{letter}", answer)

**22.Q7 — Une porte, c'est une multiplication**

In [ ]:
# 22.Q7: the values you found on paper
answer_22_Q7a = ...  # a) a whole number
answer_22_Q7b = ...  # b) the letter of your choice, e.g. "E"
answer_22_Q7c = ...  # c) True or False
answer_22_Q7d = ...  # d) True or False
answer_22_Q7e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_22_Q7a, answer_22_Q7b, answer_22_Q7c, answer_22_Q7d, answer_22_Q7e]):
    wb.check(f"22.Q7{letter}", answer)

**22.Q8 — Oublier, se souvenir, sélectionner**

In [ ]:
# 22.Q8: the values you found on paper
answer_22_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_22_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_22_Q8c = ...  # c) True or False
answer_22_Q8d = ...  # d) the letter of your choice, e.g. "E"
answer_22_Q8e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_22_Q8a, answer_22_Q8b, answer_22_Q8c, answer_22_Q8d, answer_22_Q8e]):
    wb.check(f"22.Q8{letter}", answer)

**22.Q9 — LSTM ou GRU ?**

In [ ]:
# 22.Q9: the values you found on paper
answer_22_Q9a = ...  # a) a whole number
answer_22_Q9b = ...  # b) True or False
answer_22_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_22_Q9d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_22_Q9a, answer_22_Q9b, answer_22_Q9c, answer_22_Q9d]):
    wb.check(f"22.Q9{letter}", answer)

**22.Q10 — One-to-many, many-to-one, many-to-many**

In [ ]:
# 22.Q10: the values you found on paper
answer_22_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_22_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_22_Q10c = ...  # c) the letter of your choice, e.g. "E"
answer_22_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_22_Q10e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_22_Q10a, answer_22_Q10b, answer_22_Q10c, answer_22_Q10d, answer_22_Q10e]):
    wb.check(f"22.Q10{letter}", answer)

**22.Q11 — RNN profond et bidirectionnel : quand et à quel prix ?**

In [ ]:
# 22.Q11: the values you found on paper
answer_22_Q11a = ...  # a) True or False
answer_22_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_22_Q11c = ...  # c) the letter of your choice, e.g. "E"
answer_22_Q11d = ...  # d) True or False
answer_22_Q11e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_22_Q11a, answer_22_Q11b, answer_22_Q11c, answer_22_Q11d, answer_22_Q11e]):
    wb.check(f"22.Q11{letter}", answer)

**22.Q12 — 🕰️ Transformers, RNN légers et modèles à espace d'états**

In [ ]:
# 22.Q12: the values you found on paper
answer_22_Q12a = ...  # a) a whole number
answer_22_Q12b = ...  # b) True or False
answer_22_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_22_Q12d = ...  # d) the letter of your choice, e.g. "E"
answer_22_Q12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_22_Q12a, answer_22_Q12b, answer_22_Q12c, answer_22_Q12d, answer_22_Q12e]):
    wb.check(f"22.Q12{letter}", answer)

**22.R1 — Une convolution 1D peut-elle lire une séquence ?**

In [ ]:
# 22.R1: the values you found on paper
answer_22_R1a = ...  # a) a whole number
answer_22_R1b = ...  # b) True or False
answer_22_R1c = ...  # c) a whole number
answer_22_R1d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_22_R1a, answer_22_R1b, answer_22_R1c, answer_22_R1d]):
    wb.check(f"22.R1{letter}", answer)

**22.R2 — Gradient clipping et optimiseurs adaptatifs**

In [ ]:
# 22.R2: the values you found on paper
answer_22_R2a = ...  # a) a whole number
answer_22_R2b = ...  # b) a number, 2 decimals
answer_22_R2c = ...  # c) True or False
answer_22_R2d = ...  # d) the letter of your choice, e.g. "E"
answer_22_R2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_22_R2a, answer_22_R2b, answer_22_R2c, answer_22_R2d, answer_22_R2e]):
    wb.check(f"22.R2{letter}", answer)

**22.R3 — Ajuster le scaler sur le train seulement**

In [ ]:
# 22.R3: the values you found on paper
answer_22_R3a = ...  # a) True or False
answer_22_R3b = ...  # b) the letter of your choice, e.g. "E"
answer_22_R3c = ...  # c) a number, 2 decimals
answer_22_R3d = ...  # d) True or False
answer_22_R3e = ...  # e) a word as a string, e.g. "fit"

for letter, answer in zip("abcde", [answer_22_R3a, answer_22_R3b, answer_22_R3c, answer_22_R3d, answer_22_R3e]):
    wb.check(f"22.R3{letter}", answer)

**Ex 22.1 — Combien de fenêtres ? Températures, taches solaires et Holmes**

In [ ]:
# Ex 22.1: the values you found on paper
answer_22_1a = ...  # a) a whole number
answer_22_1b = ...  # b) a whole number
answer_22_1c = ...  # c) a whole number
answer_22_1d = ...  # d) a whole number
answer_22_1e = ...  # e) a whole number
answer_22_1f = ...  # f) a whole number
answer_22_1g = ...  # g) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefg", [answer_22_1a, answer_22_1b, answer_22_1c, answer_22_1d, answer_22_1e, answer_22_1f, answer_22_1g]):
    wb.check(f"22.1{letter}", answer)

**Ex 22.2 — Dérouler à la main une cellule RNN scalaire sur trois pas**

In [ ]:
# Ex 22.2: the values you found on paper
answer_22_2a = ...  # a) a number, 4 decimals
answer_22_2b = ...  # b) a number, 4 decimals
answer_22_2c = ...  # c) a number, 4 decimals
answer_22_2d = ...  # d) a number, 4 decimals
answer_22_2e = ...  # e) a number, 4 decimals
answer_22_2f = ...  # f) a number, 2 decimals

for letter, answer in zip("abcdef", [answer_22_2a, answer_22_2b, answer_22_2c, answer_22_2d, answer_22_2e, answer_22_2f]):
    wb.check(f"22.2{letter}", answer)

**Ex 22.3 — Compter les poids : la cellule à 69 poids du livre, puis nn.RNN et nn.LSTM**

In [ ]:
# Ex 22.3: the values you found on paper
answer_22_3a = ...  # a) a whole number
answer_22_3b = ...  # b) a whole number
answer_22_3c = ...  # c) a whole number
answer_22_3d = ...  # d) a whole number
answer_22_3e = ...  # e) a whole number
answer_22_3f = ...  # f) a whole number
answer_22_3g = ...  # g) a whole number
answer_22_3h = ...  # h) a whole number
answer_22_3i = ...  # i) a whole number

for letter, answer in zip("abcdefghi", [answer_22_3a, answer_22_3b, answer_22_3c, answer_22_3d, answer_22_3e, answer_22_3f, answer_22_3g, answer_22_3h, answer_22_3i]):
    wb.check(f"22.3{letter}", answer)

**Ex 22.4 — Un pas de LSTM à la main**

In [ ]:
# Ex 22.4: the values you found on paper
answer_22_4a = ...  # a) a number, 4 decimals
answer_22_4b = ...  # b) a number, 4 decimals
answer_22_4c = ...  # c) a number, 4 decimals
answer_22_4d = ...  # d) a number, 4 decimals
answer_22_4e = ...  # e) a number, 4 decimals
answer_22_4f = ...  # f) a number, 4 decimals
answer_22_4g = ...  # g) a number, 4 decimals
answer_22_4h = ...  # h) a number, 4 decimals
answer_22_4i = ...  # i) a shape as a string, e.g. "(3, 5, 1)"

for letter, answer in zip("abcdefghi", [answer_22_4a, answer_22_4b, answer_22_4c, answer_22_4d, answer_22_4e, answer_22_4f, answer_22_4g, answer_22_4h, answer_22_4i]):
    wb.check(f"22.4{letter}", answer)

**Ex 22.5 — Softmax avec température à la main**

In [ ]:
# Ex 22.5: the values you found on paper
answer_22_5a = ...  # a) a number, 4 decimals
answer_22_5b = ...  # b) a number, 4 decimals
answer_22_5c = ...  # c) a number, 4 decimals
answer_22_5d = ...  # d) a number, 4 decimals
answer_22_5e = ...  # e) a number, 4 decimals
answer_22_5f = ...  # f) a number, 2 decimals
answer_22_5g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_22_5a, answer_22_5b, answer_22_5c, answer_22_5d, answer_22_5e, answer_22_5f, answer_22_5g]):
    wb.check(f"22.5{letter}", answer)

**Ex 22.8 — Fermi : les 17 millions de poids du CNN-LSTM et le prix d'une epoch**

In [ ]:
# Ex 22.8: the values you found on paper
answer_22_8a = ...  # a) a whole number
answer_22_8b = ...  # b) a whole number
answer_22_8c = ...  # c) a whole number
answer_22_8d = ...  # d) a whole number
answer_22_8e = ...  # e) a whole number
answer_22_8f = ...  # f) a whole number
answer_22_8g = ...  # g) a whole number
answer_22_8h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_22_8a, answer_22_8b, answer_22_8c, answer_22_8d, answer_22_8e, answer_22_8f, answer_22_8g, answer_22_8h]):
    wb.check(f"22.8{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire les équations d'une cellule RNN, LSTM et GRU, et compter leurs paramètres ?
2. Sais-tu expliquer pourquoi le gradient d'un RNN s'évanouit ou explose, et ce que change le chemin additif du LSTM ?
3. Sais-tu préparer une série temporelle sans fuite, et régler la température d'un générateur ?

**Pour aller plus loin** : le billet de C. Olah sur les LSTM et celui d'A. Karpathy sur les RNN, cités dans la fiche. La suite : le ch. 23, Keras, traduit en PyTorch.

*Les exercices de code 22.12 à 22.29 (parties A à D, tes librairies `sequences.py` et `rnn.py` et les RNN PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 22` pour obtenir le notebook complet.*